# Merged Win Pct Validation

This notebook verifies that `merged_win_pct.json` contains every fighter and that each fighter has matchup entries for every other fighter.

## Section 1: Load Source Files

In [7]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

base_dir = Path.cwd()
repo_dir = base_dir if (base_dir / "input").exists() else Path(r"C:\Users\Moudimash99\Documents\GitHub\unmatched_matcher")
fighters_path = repo_dir / "input" / "fighters.json"
merged_win_pct_path = repo_dir / "input" / "merged_win_pct.json"

with fighters_path.open("r", encoding="utf-8") as f:
    fighters_data = json.load(f)

with merged_win_pct_path.open("r", encoding="utf-8") as f:
    merged_win_pct = json.load(f)

print(f"Loaded fighters from: {fighters_path}")
print(f"Loaded merged win pct from: {merged_win_pct_path}")
print(f"Fighter records: {len(fighters_data.get('fighters', []))}")
print(f"Top-level merged fighter entries: {len(merged_win_pct)}")
print("Top-level merged keys sample:", list(merged_win_pct.keys())[:10])

Loaded fighters from: C:\Users\Moudimash99\Documents\GitHub\unmatched_matcher\input\fighters.json
Loaded merged win pct from: C:\Users\Moudimash99\Documents\GitHub\unmatched_matcher\input\merged_win_pct.json
Fighter records: 61
Top-level merged fighter entries: 72
Top-level merged keys sample: ['bullseye', 'invisible_man', 'loki', 'little_red', 'buffy', 'ciri', 'winter_soldier', 'beowulf', 'she_hulk', 'luke_cage']


## Section 2: Extract and Normalize Fighter IDs

In [8]:
def normalize_fighter_id(value):
    if value is None:
        return None
    normalized = str(value).strip().lower()
    normalized = re.sub(r"\s+", "_", normalized)
    normalized = re.sub(r"[^a-z0-9_]+", "", normalized)
    normalized = re.sub(r"_+", "_", normalized).strip("_")
    return normalized or None

fighter_records = fighters_data.get("fighters", [])
raw_fighter_ids = [fighter.get("id") for fighter in fighter_records]
normalized_fighter_ids = [normalize_fighter_id(fighter_id) for fighter_id in raw_fighter_ids]
canonical_fighter_ids = [fighter_id for fighter_id in normalized_fighter_ids if fighter_id is not None]
canonical_fighter_set = set(canonical_fighter_ids)

normalized_duplicates = sorted({fighter_id for fighter_id in canonical_fighter_ids if canonical_fighter_ids.count(fighter_id) > 1})

print(f"Canonical fighter ids: {len(canonical_fighter_set)}")
print(f"Duplicate normalized ids: {len(normalized_duplicates)}")
if normalized_duplicates:
    print("Duplicate ids:", normalized_duplicates)

canonical_fighter_df = pd.DataFrame({"fighter_id": canonical_fighter_ids})
display(canonical_fighter_df.head(20))

Canonical fighter ids: 60
Duplicate normalized ids: 1
Duplicate ids: ['bruce_lee']


,fighter_id
0,king_arthur
1,alice
2,medusa
3,sinbad
4,robin_hood
5,bigfoot
6,bruce_lee
7,robert_muldoon
8,raptors
9,dracula


## Section 3: Check Fighter Coverage in the Merged Win Pct File

In [9]:
merged_top_level_ids = {normalize_fighter_id(fighter_id) for fighter_id in merged_win_pct.keys()}
merged_top_level_ids = {fighter_id for fighter_id in merged_top_level_ids if fighter_id is not None}

missing_top_level_fighters = sorted(canonical_fighter_set - merged_top_level_ids)
unexpected_top_level_fighters = sorted(merged_top_level_ids - canonical_fighter_set)

print(f"Missing top-level fighters: {len(missing_top_level_fighters)}")
print(f"Unexpected top-level fighters: {len(unexpected_top_level_fighters)}")

coverage_df = pd.DataFrame({
    "fighter_id": sorted(canonical_fighter_set),
    "present_in_merged": [fighter_id in merged_top_level_ids for fighter_id in sorted(canonical_fighter_set)],
})
display(coverage_df)

if missing_top_level_fighters:
    print("Missing top-level fighter ids:", missing_top_level_fighters)
if unexpected_top_level_fighters:
    print("Unexpected top-level fighter ids:", unexpected_top_level_fighters)

Missing top-level fighters: 0
Unexpected top-level fighters: 12


,fighter_id,present_in_merged
0,achilles,True
1,alice,True
2,ancient_leshen,True
3,angel,True
4,annie_christmas,True
5,beowulf,True
6,bigfoot,True
7,black_panther,True
8,black_widow,True
9,bloody_mary,True


Unexpected top-level fighter ids: ['blackbeard', 'chupacabra', 'donatello', 'krang', 'leonardo', 'loki', 'martian_invader', 'michelangelo', 'mothman', 'pandora', 'raphael', 'shredder']


## Section 4: Verify Opponent Coverage for Every Fighter

In [10]:
all_expected_pairs = []
missing_pair_entries = []

for fighter_id in sorted(canonical_fighter_set):
    opponent_map = merged_win_pct.get(fighter_id)
    if not isinstance(opponent_map, dict):
        opponent_map = {}

    merged_opponent_ids = {normalize_fighter_id(opponent_id) for opponent_id in opponent_map.keys()}
    merged_opponent_ids = {opponent_id for opponent_id in merged_opponent_ids if opponent_id is not None}

    expected_opponents = canonical_fighter_set
    all_expected_pairs.append({
        "fighter_id": fighter_id,
        "expected_opponents": len(expected_opponents),
        "present_opponents": len(merged_opponent_ids & expected_opponents),
        "missing_opponents": len(expected_opponents - merged_opponent_ids),
    })

    for opponent_id in sorted(expected_opponents):
        if opponent_id not in merged_opponent_ids:
            missing_pair_entries.append({
                "fighter_id": fighter_id,
                "opponent_id": opponent_id,
                "issue": "missing opponent entry",
            })

opponent_coverage_df = pd.DataFrame(all_expected_pairs)
display(opponent_coverage_df.head(20))

print(f"Total missing pair entries: {len(missing_pair_entries)}")

,fighter_id,expected_opponents,present_opponents,missing_opponents
0,achilles,60,60,0
1,alice,60,60,0
2,ancient_leshen,60,60,0
3,angel,60,60,0
4,annie_christmas,60,60,0
5,beowulf,60,60,0
6,bigfoot,60,60,0
7,black_panther,60,60,0
8,black_widow,60,60,0
9,bloody_mary,60,60,0


Total missing pair entries: 0


## Section 5: List Missing Fighters and Missing Pair Entries

In [11]:
missing_fighters_df = pd.DataFrame({"fighter_id": missing_top_level_fighters})
missing_pairs_df = pd.DataFrame(missing_pair_entries)

print("Missing fighters table:")
display(missing_fighters_df)

print("Missing pair entries table:")
display(missing_pairs_df.head(200))

print(f"Missing fighters rows: {len(missing_fighters_df)}")
print(f"Missing pair rows: {len(missing_pairs_df)}")

Missing fighters table:


,fighter_id


Missing pair entries table:


""


Missing fighters rows: 0
Missing pair rows: 0


## Section 6: Summarize Validation Results

In [12]:
total_fighters = len(canonical_fighter_set)
matched_fighters = total_fighters - len(missing_top_level_fighters)
expected_pairings = total_fighters * total_fighters
missing_pairings = len(missing_pair_entries)
matched_pairings = expected_pairings - missing_pairings
validation_passed = (
    len(missing_top_level_fighters) == 0
    and len(unexpected_top_level_fighters) == 0
    and missing_pairings == 0
)

summary_df = pd.DataFrame([
    {"metric": "total_fighters", "value": total_fighters},
    {"metric": "matched_fighters", "value": matched_fighters},
    {"metric": "missing_fighters", "value": len(missing_top_level_fighters)},
    {"metric": "unexpected_top_level_fighters", "value": len(unexpected_top_level_fighters)},
    {"metric": "expected_pairings", "value": expected_pairings},
    {"metric": "matched_pairings", "value": matched_pairings},
    {"metric": "missing_pairings", "value": missing_pairings},
    {"metric": "validation_passed", "value": validation_passed},
])

display(summary_df)
print("Validation passed:" if validation_passed else "Validation failed:", validation_passed)

,metric,value
0,total_fighters,60
1,matched_fighters,60
2,missing_fighters,0
3,unexpected_top_level_fighters,12
4,expected_pairings,3600
5,matched_pairings,3600
6,missing_pairings,0
7,validation_passed,False


Validation failed: False


In [ ]:
flag_rows = []

for fighter_id in sorted(canonical_fighter_set):
    opponent_map = merged_win_pct.get(fighter_id, {})
    if not isinstance(opponent_map, dict):
        opponent_map = {}

    expected_opponents = sorted(canonical_fighter_set)
    values = [opponent_map.get(opponent_id) for opponent_id in expected_opponents]
    minus_two_count = sum(value == -2 for value in values)
    total_expected = len(expected_opponents)

    flag_rows.append({
        "fighter_id": fighter_id,
        "minus_two_count": minus_two_count,
        "total_expected": total_expected,
        "all_minus_two": minus_two_count == total_expected,
    })

flag_df = pd.DataFrame(flag_rows).sort_values(
    ["all_minus_two", "minus_two_count", "fighter_id"],
    ascending=[False, False, True],
)

display(flag_df)

flagged_fighters = flag_df.loc[flag_df["all_minus_two"], "fighter_id"].tolist()
print(f"Fighters with all -2 values: {len(flagged_fighters)}")
if flagged_fighters:
    print("Flagged fighters:", flagged_fighters)



,fighter_id,minus_two_count,total_expected,all_minus_two
35,muhammad_ali,20,60,False
56,willow,9,60,False
46,spike,7,60,False
22,eredin,6,60,False
3,angel,5,60,False
13,ciri,5,60,False
23,geralt,5,60,False
2,ancient_leshen,4,60,False
16,deadpool,4,60,False
28,jekyll_hyde,4,60,False


Fighters with all -2 values: 0


In [17]:
# if there is a fighter that has -2 values with more than 60% also print it as a log
high_minus_two_df = flag_df.loc[
    (flag_df["total_expected"] > 0)
    & ((flag_df["minus_two_count"] / flag_df["total_expected"]) > 0.60)
].copy()

if not high_minus_two_df.empty:
    high_minus_two_df["minus_two_ratio"] = (
        high_minus_two_df["minus_two_count"] / high_minus_two_df["total_expected"]
    )
    print(f"Fighters with >60% -2 values: {len(high_minus_two_df)}")
    for _, row in high_minus_two_df.sort_values("minus_two_ratio", ascending=False).iterrows():
        print(
            f"[HIGH -2] {row['fighter_id']}: "
            f"{row['minus_two_count']}/{row['total_expected']} "
            f"({row['minus_two_ratio']:.1%})"
        )
else:
    print("nothing is printed")


nothing is printed
